In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
GOLD_DB = PROJECT_ROOT / "data" / "gold" / "fraud_dev.duckdb"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def q(sql):
    """Run a query on the Gold warehouse (read-only) and return a DataFrame."""
    with duckdb.connect(str(GOLD_DB), read_only=True) as con:
        return con.sql(sql).df()


print("Gold file exists:", GOLD_DB.exists())
q("""
    select count(*)      as n_rows,
           sum(is_fraud) as n_fraud
    from fact_transactions
""")

Gold file exists: True


,n_rows,n_fraud
0,1852394,9651.0


In [2]:
monthly = q("""
    select strftime(trans_ts, '%Y-%m')                       as month,
           count(*)                                          as n_txn,
           cast(sum(is_fraud) as integer)                    as n_fraud,
           round(100 * avg(is_fraud), 3)                     as fraud_pct,
           count(distinct case when is_fraud = 1
                               then card_key end)            as fraud_cards
    from fact_transactions
    group by 1
    order by 1
""")

print("total rows :", monthly["n_txn"].sum())
print("total fraud:", monthly["n_fraud"].sum())
monthly

total rows : 1852394
total fraud: 9651


,month,n_txn,n_fraud,fraud_pct,fraud_cards
0,2019-01,52525,506,0.963,50
1,2019-02,49866,517,1.037,53
2,2019-03,70939,494,0.696,49
3,2019-04,68078,376,0.552,41
4,2019-05,72532,408,0.563,42
5,2019-06,86064,354,0.411,35
6,2019-07,86596,331,0.382,36
7,2019-08,87359,382,0.437,39
8,2019-09,70652,418,0.592,44
9,2019-10,68758,454,0.660,50


In [3]:
from src.models.split import TRAIN, GAP, VALIDATION, TEST, assign_split

tx = q("select trans_ts, is_fraud, card_key from fact_transactions")
tx["split"] = assign_split(tx["trans_ts"])

fraud_rows = tx[tx["is_fraud"] == 1]
split_summary = pd.DataFrame({
    "n_txn": tx.groupby("split").size(),
    "n_fraud": tx.groupby("split")["is_fraud"].sum(),
    "fraud_cards": fraud_rows.groupby("split")["card_key"].nunique(),
}).reindex([TRAIN, GAP, VALIDATION, TEST])
split_summary["fraud_pct"] = (100 * split_summary["n_fraud"] / split_summary["n_txn"]).round(3)

print("total rows :", split_summary["n_txn"].sum())
print("total fraud:", split_summary["n_fraud"].sum())
split_summary

total rows : 1852394
total fraud: 9651


,n_txn,n_fraud,fraud_cards,fraud_pct
split,,,,
train,1306680,7528,762,0.576
gap,36006,208,23,0.578
validation,228187,979,102,0.429
test,281521,936,94,0.332


In [4]:
# Which splits does each fraud card appear in?
card_splits = (
    fraud_rows.groupby("card_key")["split"]
    .agg(lambda s: " + ".join(sorted(set(s))))
)
print("fraud cards in total:", card_splits.size)
card_splits.value_counts()

fraud cards in total: 976


split
train               762
validation           98
test                 93
gap                  18
gap + validation      4
gap + test            1
Name: count, dtype: int64

In [5]:
for table in ["fact_transactions", "dim_card", "dim_merchant", "dim_location", "dim_date"]:
    cols = q(f"describe {table}")[["column_name", "column_type"]]
    print(f"\n{table}:")
    print(", ".join(f"{c} ({t})" for c, t in cols.itertuples(index=False)))


fact_transactions:
trans_num (VARCHAR), card_key (VARCHAR), merchant_key (VARCHAR), location_key (VARCHAR), date_key (INTEGER), trans_ts (TIMESTAMP), amt (DOUBLE), is_fraud (TINYINT), merch_lat (DOUBLE), merch_long (DOUBLE), distance_km (DOUBLE), is_public_holiday_in_state (BOOLEAN), _batch_date (DATE)

dim_card:
card_key (VARCHAR), card_hash (VARCHAR), gender (VARCHAR), birth_year (SMALLINT), job (VARCHAR)

dim_merchant:
merchant_key (VARCHAR), merchant (VARCHAR), category (VARCHAR), mcc (VARCHAR), mcc_description (VARCHAR), channel (VARCHAR)

dim_location:
location_key (VARCHAR), zip (VARCHAR), city (VARCHAR), state (VARCHAR), city_pop (BIGINT), population (BIGINT), median_household_income (BIGINT), has_income (BOOLEAN), median_age (DOUBLE), land_area_m2 (BIGINT), water_area_m2 (BIGINT), centroid_lat (DOUBLE), centroid_long (DOUBLE)

dim_date:
date_key (INTEGER), calendar_date (DATE), calendar_year (BIGINT), calendar_month (BIGINT), day_of_month (BIGINT), day_of_week (BIGINT), day_n

In [6]:
import time
from src.models.features import build_features, load_model_frame

t0 = time.perf_counter()
frame = load_model_frame(GOLD_DB)
features = build_features(frame)
print(f"built in {time.perf_counter() - t0:.1f} s")

print("rows :", len(frame), "| fraud:", int(frame["is_fraud"].sum()))
print("missing secs_since_prev    :", features["secs_since_prev"].isna().sum())
print("missing amt_vs_card_median :", features["amt_vs_card_median"].isna().sum())
print("missing median_income      :", features["median_household_income"].isna().sum())

high = features["amt_vs_card_median"] > 10
print("ratio > 10 :", int(high.sum()), "rows |", int(frame.loc[high, "is_fraud"].sum()), "fraud")

features.describe().T.round(2)

built in 6.0 s
rows : 1852394 | fraud: 9651
missing secs_since_prev    : 999
missing amt_vs_card_median : 4995
missing median_income      : 124334
ratio > 10 : 23001 rows | 4355 fraud


,count,mean,std,min,25%,50%,75%,max
amt,1852394.0,70.06,159.25,1.00,9.64,47.45,83.10,28948.90
hour,1852394.0,12.81,6.82,0.00,7.00,14.00,19.00,23.00
is_night,1852394.0,0.23,0.42,0.00,0.00,0.00,0.00,1.00
day_of_week,1852394.0,2.97,2.20,0.00,1.00,3.00,5.00,6.00
secs_since_prev,1851395.0,30939.97,45328.96,1.00,5699.00,15695.00,38135.00,1341471.00
amt_vs_card_median,1847399.0,1.61,4.22,0.01,0.26,1.00,1.88,855.71
card_txn_1h,1852394.0,0.20,0.48,0.00,0.00,0.00,0.00,8.00
card_txn_24h,1852394.0,4.11,3.26,0.00,2.00,3.00,6.00,35.00
card_prev_txn_count,1852394.0,1298.90,969.45,0.00,509.00,1109.00,1917.00,4391.00
age,1852394.0,46.21,17.40,14.00,33.00,44.00,57.00,96.00


In [7]:
key = ["card_key", "trans_ts"]
dup_mask = frame.duplicated(key, keep=False)
dups = frame.loc[dup_mask, key + ["trans_num", "amt", "category", "is_fraud"]].copy()
dups["card_key"] = dups["card_key"].str[:8]
dups = dups.sort_values(key)

groups = dups.groupby(key)
print("rows involved :", len(dups))
print("groups        :", groups.ngroups)
print("rows per group:", groups.size().value_counts().to_dict())
print("fraud rows    :", int(dups["is_fraud"].sum()))
print("same amount in group:", int((groups["amt"].nunique() == 1).sum()), "groups")
dups.head(20)

rows involved : 88
groups        : 44
rows per group: {2: 44}
fraud rows    : 0
same amount in group: 0 groups


,card_key,trans_ts,trans_num,amt,category,is_fraud
807247,07e04be9,2019-12-06 08:43:10,4b0e0c217ffd7da36b4c1b514506b261,46.96,gas_transport,0
807248,07e04be9,2019-12-06 08:43:10,f065803df4963911996992a13be9e730,6.62,shopping_net,0
203800,16c3c4aa,2019-04-14 16:33:06,aa86fe954b3b2eea97fce2c199d93355,6.16,misc_pos,0
203801,16c3c4aa,2019-04-14 16:33:06,d12aa840819477ffcc2104cc13523bdd,2.42,shopping_pos,0
1378905,1d52a6cc,2020-07-20 01:51:36,02d8dd13601ea982d7954b225807fe1a,91.50,gas_transport,0
1378906,1d52a6cc,2020-07-20 01:51:36,c19a3c1f6c25d09be58d9b9a1b1e9472,37.79,grocery_net,0
1438721,1f079572,2020-08-10 02:54:51,3f4186c36e810790b42cbb6f60d6e79a,142.44,shopping_pos,0
1438722,1f079572,2020-08-10 02:54:51,a9fc75135e5d99b98ddde96180e7791b,146.47,shopping_pos,0
26757,25c0995b,2019-01-16 05:51:27,a0e8115ba818d5bfd945bdd0fc7d99d8,44.74,grocery_pos,0
26758,25c0995b,2019-01-16 05:51:27,b235d4c2ccc00363554c8d998efa9f30,93.79,gas_transport,0


In [8]:
from src.models.evaluate import alert_metrics, card_bootstrap, rule_baseline

data = frame[["card_key", "trans_ts", "is_fraud", "gender"]].copy()
data["split"] = assign_split(data["trans_ts"])
rule = rule_baseline(features)

for name in [TRAIN, VALIDATION]:
    m = data["split"] == name
    res = alert_metrics(data.loc[m, "is_fraud"], rule[m], features.loc[m, "amt"])
    days = data.loc[m, "trans_ts"].dt.normalize().nunique()
    print(f"\n{name}: {days} days, {res['alerts'] / days:.1f} alerts/day")
    for k, v in res.items():
        print(f"  {k:15s} {v:,.4f}" if isinstance(v, float) else f"  {k:15s} {v:,}")

# 95% interval for the rule's recall on validation, resampling whole cards
val = (data["split"] == VALIDATION).to_numpy()
y_val = data.loc[val, "is_fraud"].to_numpy()
r_val = rule[val].to_numpy()
amt_val = features.loc[val, "amt"].to_numpy()
ci = card_bootstrap(
    data.loc[val, "card_key"],
    lambda idx: alert_metrics(y_val[idx], r_val[idx], amt_val[idx])["recall"],
)
print(f"\nvalidation recall {ci['estimate']:.3f}  95% CI [{ci['low']:.3f}, {ci['high']:.3f}]")


train: 539 days, 65.0 alerts/day
  alerts          35,023
  fraud_caught    4,874
  recall          0.6474
  precision       0.1392
  dollars_caught  3,342,187.3300
  dollar_recall   0.8354

validation: 85 days, 68.6 alerts/day
  alerts          5,832
  fraud_caught    629
  recall          0.6425
  precision       0.1079
  dollars_caught  432,720.8900
  dollar_recall   0.8382

validation recall 0.642  95% CI [0.608, 0.679]


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.models.evaluate import pr_auc, top_n_mask
from src.models.features import CARD_HISTORY_FEATURES, CATEGORICAL_FEATURES, FEATURE_COLUMNS

LOG_FEATURES = ["amt", "secs_since_prev", "amt_vs_card_median", "city_pop"]
numeric = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES + CARD_HISTORY_FEATURES]
plain = [c for c in numeric if c not in LOG_FEATURES]

prep = ColumnTransformer([
    ("log", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                          FunctionTransformer(np.log1p), StandardScaler()), LOG_FEATURES),
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                          StandardScaler()), plain),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
logit = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])

trn = (data["split"] == TRAIN).to_numpy()
t0 = time.perf_counter()
logit.fit(features[trn], data.loc[trn, "is_fraud"])
print(f"fitted in {time.perf_counter() - t0:.1f} s")

p_val = logit.predict_proba(features[val])[:, 1]
n_alerts = int(r_val.sum())
print(f"\nvalidation PR-AUC   logit {pr_auc(y_val, p_val):.4f} | rule {pr_auc(y_val, r_val):.4f}")

res = alert_metrics(y_val, top_n_mask(p_val, n_alerts), amt_val)
print(f"\nlogit at the rule's workload ({n_alerts:,} alerts):")
for k, v in res.items():
    print(f"  {k:15s} {v:,.4f}" if isinstance(v, float) else f"  {k:15s} {v:,}")

cards_val = data.loc[val, "card_key"]
ci = card_bootstrap(
    cards_val,
    lambda idx: alert_metrics(
        y_val[idx], top_n_mask(p_val[idx], int(r_val[idx].sum())), amt_val[idx]
    )["recall"],
)
print(f"\nlogit recall {ci['estimate']:.3f}  95% CI [{ci['low']:.3f}, {ci['high']:.3f}]")

fitted in 4.1 s

validation PR-AUC   logit 0.4285 | rule 0.0708

logit at the rule's workload (5,832 alerts):
  alerts          5,832
  fraud_caught    672
  recall          0.6864
  precision       0.1152
  dollars_caught  467,879.1700
  dollar_recall   0.9063

logit recall 0.686  95% CI [0.645, 0.727]


In [10]:
def paired_diff(idx, key):
    """Logit minus rule on the SAME resampled cards, at equal workload."""
    n = int(r_val[idx].sum())
    logit_m = alert_metrics(y_val[idx], top_n_mask(p_val[idx], n), amt_val[idx])
    rule_m = alert_metrics(y_val[idx], r_val[idx], amt_val[idx])
    return logit_m[key] - rule_m[key]

for key in ["recall", "precision", "dollar_recall"]:
    d = card_bootstrap(cards_val, lambda idx: paired_diff(idx, key))
    verdict = "real difference" if d["low"] > 0 or d["high"] < 0 else "not distinguishable"
    print(f"{key:14s} diff {d['estimate']:+.3f}  95% CI [{d['low']:+.3f}, {d['high']:+.3f}]  -> {verdict}")

recall         diff +0.044  95% CI [+0.024, +0.063]  -> real difference
precision      diff +0.007  95% CI [+0.004, +0.011]  -> real difference
dollar_recall  diff +0.068  95% CI [+0.044, +0.091]  -> real difference


In [11]:
from src.models.train import fit_final_lgbm, model_columns, tune_lgbm

X_train_all = features.loc[trn]
y_train = data.loc[trn, "is_fraud"].to_numpy()
ts_train = data.loc[trn, "trans_ts"]

VARIANTS = {
    "A_plain":            {"weighted": False, "history": False},
    "B_plain_history":    {"weighted": False, "history": True},
    "C_weighted":         {"weighted": True,  "history": False},
    "D_weighted_history": {"weighted": True,  "history": True},
}

tuning = {}
for name, v in VARIANTS.items():
    cols = model_columns(v["history"])
    t0 = time.perf_counter()
    tuning[name] = tune_lgbm(X_train_all[cols], y_train, ts_train, weighted=v["weighted"])
    print(f"\n{name}  ({time.perf_counter() - t0:.0f} s)")
    print(tuning[name].round(4).to_string(index=False))


A_plain  (42 s)
 num_leaves  min_child_samples  best_iteration  inner_ap
         63                500             346    0.9472
         63                 50             347    0.9462
         15                500             767    0.9457
         15                 50             228    0.9286

B_plain_history  (45 s)
 num_leaves  min_child_samples  best_iteration  inner_ap
         63                500             190    0.9426
         63                 50             200    0.9417
         15                500             592    0.9360
         15                 50             846    0.9355

C_weighted  (67 s)
 num_leaves  min_child_samples  best_iteration  inner_ap
         63                 50             293    0.9450
         63                500             453    0.9441
         15                 50             834    0.9388
         15                500             776    0.9366

D_weighted_history  (53 s)
 num_leaves  min_child_samples  best_iteration  inner_a

In [12]:
preds = {"rule": r_val.astype(float), "logit": p_val}
final_models = {}

for name, v in VARIANTS.items():
    cols = model_columns(v["history"])
    row = tuning[name].iloc[0]
    best = {k: int(row[k]) for k in ["num_leaves", "min_child_samples", "best_iteration"]}
    t0 = time.perf_counter()
    model = fit_final_lgbm(X_train_all[cols], y_train, v["weighted"], best)
    final_models[name] = (model, cols)
    preds[name] = model.predict_proba(features.loc[val, cols])[:, 1]
    print(f"{name}: refit on full train in {time.perf_counter() - t0:.0f} s")

summary = {}
for name, p in preds.items():
    m = alert_metrics(y_val, top_n_mask(p, n_alerts), amt_val)
    summary[name] = {
        "pr_auc": pr_auc(y_val, p),
        "recall": m["recall"],
        "precision": m["precision"],
        "dollar_recall": m["dollar_recall"],
        "fraud_caught": m["fraud_caught"],
    }
print(f"\nvalidation, {n_alerts:,} alerts for every model:")
pd.DataFrame(summary).T.round(4)

A_plain: refit on full train in 9 s
B_plain_history: refit on full train in 5 s
C_weighted: refit on full train in 8 s
D_weighted_history: refit on full train in 9 s

validation, 5,832 alerts for every model:


,pr_auc,recall,precision,dollar_recall,fraud_caught
rule,0.0708,0.6425,0.1079,0.8382,629.0
logit,0.4285,0.6864,0.1152,0.9063,672.0
A_plain,0.9203,0.9857,0.1655,0.9970,965.0
B_plain_history,0.9193,0.9898,0.1662,0.9995,969.0
C_weighted,0.9162,0.9888,0.1660,0.9995,968.0
D_weighted_history,0.9132,0.9847,0.1653,0.9976,964.0


In [13]:
days_val = 85
budgets = [250, 500, 1000, 2000, n_alerts]
rows = []
for b in budgets:
    for name, p in preds.items():
        if name == "rule":
            continue  # the rule has a fixed number of alerts
        m = alert_metrics(y_val, top_n_mask(p, b), amt_val)
        rows.append({"alerts": b, "per_day": round(b / days_val, 1), "model": name,
                     "recall": m["recall"], "dollar_recall": m["dollar_recall"]})
curve = pd.DataFrame(rows)

print("RECALL (share of fraud cases caught)")
print(curve.pivot(index=["alerts", "per_day"], columns="model", values="recall").round(3))
print("\nDOLLAR RECALL (share of fraud dollars caught)")
print(curve.pivot(index=["alerts", "per_day"], columns="model", values="dollar_recall").round(3))

RECALL (share of fraud cases caught)
model           A_plain  B_plain_history  C_weighted  D_weighted_history  logit
alerts per_day                                                                 
250    2.9        0.255            0.255       0.255               0.255  0.207
500    5.9        0.511            0.511       0.508               0.508  0.344
1000   11.8       0.855            0.853       0.848               0.846  0.461
2000   23.5       0.947            0.948       0.951               0.945  0.587
5832   68.6       0.986            0.990       0.989               0.985  0.686

DOLLAR RECALL (share of fraud dollars caught)
model           A_plain  B_plain_history  C_weighted  D_weighted_history  logit
alerts per_day                                                                 
250    2.9        0.257            0.293       0.245               0.275  0.355
500    5.9        0.570            0.568       0.571               0.581  0.554
1000   11.8       0.946            0

In [14]:
    BUDGET = 1000

def paired(idx, a, b, metric):
    """Model a minus model b on the SAME resampled cards."""
    y, amt = y_val[idx], amt_val[idx]
    pa, pb = preds[a][idx], preds[b][idx]
    if metric == "pr_auc":
        return pr_auc(y, pa) - pr_auc(y, pb)
    ma = alert_metrics(y, top_n_mask(pa, BUDGET), amt)
    mb = alert_metrics(y, top_n_mask(pb, BUDGET), amt)
    return ma[metric] - mb[metric]

t0 = time.perf_counter()
rows = []
for other in ["B_plain_history", "C_weighted", "D_weighted_history"]:
    for metric in ["pr_auc", "recall", "dollar_recall"]:
        d = card_bootstrap(cards_val, lambda idx: paired(idx, "A_plain", other, metric), n_boot=300)
        rows.append({
            "comparison": f"A - {other}", "metric": metric,
            "diff": d["estimate"], "low": d["low"], "high": d["high"],
            "verdict": "real" if d["low"] > 0 or d["high"] < 0 else "not distinguishable",
        })
print(f"done in {time.perf_counter() - t0:.0f} s")
pd.DataFrame(rows).round(4)

done in 139 s


,comparison,metric,diff,low,high,verdict
0,A - B_plain_history,pr_auc,0.0010,-0.0047,0.0075,not distinguishable
1,A - B_plain_history,recall,0.0020,-0.0103,0.0128,not distinguishable
2,A - B_plain_history,dollar_recall,0.0068,-0.0074,0.0178,not distinguishable
3,A - C_weighted,pr_auc,0.0042,-0.0000,0.0091,not distinguishable
4,A - C_weighted,recall,0.0072,-0.0076,0.0153,not distinguishable
5,A - C_weighted,dollar_recall,0.0010,-0.0179,0.0074,not distinguishable
6,A - D_weighted_history,pr_auc,0.0071,-0.0047,0.0186,not distinguishable
7,A - D_weighted_history,recall,0.0092,-0.0074,0.0254,not distinguishable
8,A - D_weighted_history,dollar_recall,0.0062,-0.0067,0.0279,not distinguishable
